In [105]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


## Langchain Integration
https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/gen_ai_hub.html#langchain-integration

### Harmonized Model Initialization
The init_llm and init_embedding_model functions allow easy initialization of langchain model interfaces in a harmonized way in generative AI hub sdk

In [106]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from gen_ai_hub.proxy.langchain.init_models import init_llm

template = """Question: {question}
    Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=['question'])
question = 'What is a supernova?'

llm = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=800
)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({'question': question})
print(response)

A supernova is a powerful and luminous explosion that occurs at the end of a star's life cycle. Let's break down the process step by step:

1. **Star's Life Cycle**: Stars are massive celestial bodies composed primarily of hydrogen and helium. They generate energy through nuclear fusion, converting hydrogen into helium in their cores. This process releases a tremendous amount of energy, which counteracts the gravitational forces trying to collapse the star.

2. **End of Fusion**: As a star exhausts its hydrogen fuel, it begins to fuse heavier elements. In massive stars, this process continues until iron is formed in the core. Iron fusion does not release energy, leading to a lack of outward pressure to balance gravity.

3. **Core Collapse**: Without the outward pressure from fusion, the core of the star collapses under its own gravity. This collapse happens extremely rapidly, in a matter of seconds.

4. **Rebound and Explosion**: The core collapse results in a shock wave that rebounds 

init_embedding_model

In [107]:
from gen_ai_hub.proxy.langchain.init_models import init_embedding_model

text = 'Every decoding is another encoding.'

embeddings = init_embedding_model('text-embedding-3-large')
response = embeddings.embed_query(text)
#print(response)


### Chat model

In [108]:
from langchain_core.prompts.chat import (
    AIMessagePromptTemplate,
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')

chat_llm = ChatOpenAI(proxy_model_name='gpt-4o', proxy_client=proxy_client)

template = 'You are a helpful assistant that translates english to Chinese.'
system_message_prompt = SystemMessagePromptTemplate.from_template(template)

example_human = HumanMessagePromptTemplate.from_template('Hi')
example_ai = AIMessagePromptTemplate.from_template('Ahoy!')
human_template = '{text}'

human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)
chat_prompt = ChatPromptTemplate.from_messages(
    [system_message_prompt, example_human, example_ai, human_message_prompt])

chain = chat_prompt | chat_llm

response = chain.invoke({'text': 'I love planking.'})
print(response.content)


我喜欢平板支撑。


### Structured model outputs

In [109]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.prompts.chat import HumanMessage
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
chat_model = ChatOpenAI(proxy_model_name="gpt-4o", proxy_client=get_proxy_client())
chat_model = chat_model.with_structured_output(method="json_schema", schema=Person, strict=True)

message = HumanMessage(content="Tell me about a person named John who is 30")
print(chat_model.invoke([message]))


name='John' age=30


## Agent

### Basic structure

#### Define tools

In [110]:
from langchain.tools import tool
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

#### Define agents

In [111]:
from langchain.agents import create_agent
from gen_ai_hub.proxy.langchain.init_models import init_llm
 
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


agent = create_agent(
    model,
    tools = [search, get_weather],
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

#### Invoke agents

In [112]:

response =agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "What is weather in Shanghai?"
            }
        ]
    }
)
print(response)

 

{'messages': [HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='6e5d6977-a882-46c5-a087-8c967baea53f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QRr0sHsBOMT2e6xgcwK3yhnE', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CoPVkpuOC7UGlTCamEpRDHkOBzSpm', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b3594-c206-73f3-91b6-0ea537a849b1-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'c

##### Optional:study on the response structure

In [113]:
print(type(response))
len(response)
response['messages']

<class 'dict'>


[HumanMessage(content='What is weather in Shanghai?', additional_kwargs={}, response_metadata={}, id='6e5d6977-a882-46c5-a087-8c967baea53f'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QRr0sHsBOMT2e6xgcwK3yhnE', 'function': {'arguments': '{"location":"Shanghai"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 78, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4a331a0222', 'id': 'chatcmpl-CoPVkpuOC7UGlTCamEpRDHkOBzSpm', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b3594-c206-73f3-91b6-0ea537a849b1-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shanghai'}, 'id': 'call_QRr0sHsB

In [114]:
messages=response['messages']
print(type(messages))
HumanMessage=messages[0]
AIMessage=messages[1]
ToolMessage=messages[2]
AIMessage_output=messages[-1]
AIMessage_output.content

<class 'list'>


'The weather in Shanghai is currently sunny with a temperature of 72°F.'

In [115]:
print(type(AIMessage))
print(json.dumps(AIMessage.additional_kwargs,indent=2))

<class 'langchain_core.messages.ai.AIMessage'>
{
  "tool_calls": [
    {
      "id": "call_QRr0sHsBOMT2e6xgcwK3yhnE",
      "function": {
        "arguments": "{\"location\":\"Shanghai\"}",
        "name": "get_weather"
      },
      "type": "function"
    }
  ],
  "refusal": null
}


In [116]:
tool_calls=AIMessage.additional_kwargs['tool_calls'][0]
print(type(tool_calls))
function=tool_calls['function']
function['name']

<class 'dict'>


'get_weather'

#### Check the message

Define a function to return message directly.

In [117]:

def invoke_agent_messages(agent, content: str):
    payload = {
        "messages": [
            {
                "role": "user", 
                "content": content
            }
        ]
    }
    response = agent.invoke(payload)
    return response["messages"]  # 若缺失会直接抛 KeyError


To check the contect in an easier way, we create a function to find out and then print out key information based on the structure of this message.

In [118]:
import json

def print_message_pairs(messages, verbose=False):
    """
    自动从消息序列中提取：
      - user_query：第一个 HumanMessage 的 content
      - tool_name：第一个 AIMessage.additional_kwargs.tool_calls[0].function.name
      - tool_output：第一个 ToolMessage 的 content
      - assistant_text：最后一个 AIMessage 的 content

    参数：
      - messages: 消息序列（包含 HumanMessage / AIMessage / ToolMessage 等）
      - verbose (bool): 
          True  -> 打印 JSON（包含四个键值）
          False -> 仅打印 assistant_text

    返回：
      - pairs (dict): 以上四个字段的字典，便于后续使用
    """
    # 安全提取工具名
    def extract_tool_name_from_ai(ai_msg):
        ak = getattr(ai_msg, "additional_kwargs", {})
        if isinstance(ak, dict):
            tool_calls = ak.get("tool_calls") or []
            if tool_calls:
                fn = tool_calls[0].get("function") or {}
                return fn.get("name")
        return None

    # 初始化
    user_query = None
    tool_name = None
    tool_output = None
    assistant_text = None

    # 1) 找第一个 HumanMessage 作为用户文本
    for m in messages:
        if m.__class__.__name__ == "HumanMessage":
            user_query = getattr(m, "content", None)
            break

    # 2) 找第一个 AIMessage 中的 tool_calls 取函数名
    for m in messages:
        if m.__class__.__name__ == "AIMessage":
            tool_name = extract_tool_name_from_ai(m)
            if tool_name:
                break

    # 3) 找第一个 ToolMessage 的输出
    for m in messages:
        if m.__class__.__name__ == "ToolMessage":
            tool_output = getattr(m, "content", None)
            break

    # 4) 找最后一个 AIMessage 的最终回复
    for m in reversed(messages):
        if m.__class__.__name__ == "AIMessage":
            assistant_text = getattr(m, "content", None)
            break

    pairs = {
        "user_query": user_query,
        "tool_name": tool_name,
        "tool_output": tool_output,
        #"assistant_text": assistant_text,
    }

    # 根据 verbose 控制打印
    if verbose:
        # 打印完整 JSON；ensure_ascii=False 支持中文直出（可按需移除）
        print(json.dumps(pairs, indent=2, ensure_ascii=False))
        print("\nAassistant reply:")
        print(assistant_text or "")
    else:
        # 仅打印最终助手回复；为防 None，做一下空串兜底
        print(assistant_text or "")




Now it is easier for us to see that, in the below case the tool [search] is not applied.

In [119]:
messages = invoke_agent_messages(agent, "Where is the location of Shanghai?")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Where is the location of Shanghai?",
  "tool_name": null,
  "tool_output": null
}

Aassistant reply:
Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea.


To let agent to use the tool, change the question closer to the tool description.

In [120]:
messages = invoke_agent_messages(agent, "Search for the location of Shanghai")
print_message_pairs(messages,verbose=True)

{
  "user_query": "Search for the location of Shanghai",
  "tool_name": "search",
  "tool_output": "Results for: location of Shanghai"
}

Aassistant reply:
Shanghai is located on the eastern coast of China, at the mouth of the Yangtze River. It is situated in the central part of the country, facing the East China Sea.


### Dynamic system prompt

For more advanced use cases where you need to modify the system prompt based on runtime context or agent state, you can use middleware.<br>
The <i>@dynamic_prompt</i> decorator creates middleware that generates system prompts based on the model request:

In [121]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent_dyn = create_agent(
    model,
    tools = [search],
    middleware=[user_role_prompt],
    context_schema=Context
)


The system prompt will be set dynamically based on context so we get different answers to the same question.

In [122]:
query="Search for the explaination of context_schemaand and mmiddleware in Langchain Agent, and then interpret them further."

In [123]:
# When set user_role as "beginner"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content":query}]},
    context={"user_role": "beginner"}
)
messages=response["messages"]
print_message_pairs(messages)

It seems I couldn't retrieve specific information from the search. However, I can provide a general explanation based on my knowledge up to October 2023.

### Context Schema in Langchain Agent

**Context Schema** typically refers to the structure or format of the data that an agent or system expects to receive or process. In the context of Langchain, which is a framework for building applications with language models, a context schema would define how information is organized and passed to the agent. This could include details like:

- **Input Types**: What kind of data the agent expects (e.g., text, numbers, JSON objects).
- **Structure**: How the data is organized (e.g., fields, keys, and values).
- **Constraints**: Any rules or limitations on the data (e.g., required fields, data types).

This schema helps ensure that the agent can correctly interpret and process the incoming data, leading to more accurate and reliable outputs.

### Middleware in Langchain Agent

**Middleware** in t

In [124]:
# When set user_role as "expert"
response = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": query}]},
    context={"user_role": "expert"}
)
messages=response["messages"]
print_message_pairs(messages)

I couldn't retrieve the specific explanations for "context_schema" and "mmiddleware" in Langchain Agent directly. However, I can provide a general interpretation based on typical usage patterns in software frameworks like Langchain.

### Context Schema in Langchain Agent

**Context Schema** typically refers to the structured format or blueprint that defines the context in which an agent operates. In the context of Langchain, which is a framework for building language model applications, a context schema might include:

- **Input Parameters**: The data or information that the agent needs to process.
- **Environment Variables**: Settings or configurations that affect how the agent operates.
- **State Information**: Details about the current state of the agent, which might include previous interactions or decisions.

The context schema ensures that the agent has all the necessary information to perform its tasks effectively and consistently. It acts as a contract between different compone

### Decorater

#### Setup: model + tools

In [125]:
from langchain_core.tools import tool
# --- Define tools ---
@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# @tool
# def tool1(query: str) -> str:
#     """A demo tool that echoes the query with a tag."""
#     return f"[query] You asked: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"



tools = [search, get_weather]

# --- Define model (replace your API key/config as needed) ---
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=4800
)

#### Define custom Context + middleware

In [126]:

from typing import TypedDict, Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware

class Context(TypedDict):
    user_preferences: dict  # {"style": "...", "verbosity": "..."}

class CustomMiddleware(AgentMiddleware):
    # (Optional) stage-specific tool restrictions:
    # tools = [tool1, tool2]

    def before_model(self, state, runtime) -> dict[str, Any] | None:
        # Read preferences from runtime context
        prefs = runtime.context.get("user_preferences", {}) or {}
        style = str(prefs.get("style", "general")).lower()
        verbosity = str(prefs.get("verbosity", "normal")).lower()

        # Base prompt
        system_prompt = "You are a helpful assistant."

        # Style-specific guidance
        if style == "technical":
            system_prompt += " Prefer precise, technical language and include implementation details."
        elif style == "casual":
            system_prompt += " Keep explanations informal, approachable, and friendly."

        # Verbosity-specific guidance
        if verbosity in ("detailed", "high"):
            system_prompt += " Provide thorough, step-by-step explanations with concrete examples."
        elif verbosity in ("brief", "low"):
            system_prompt += " Be concise and focus on key points; use short sentences and bullet points where helpful."

        # Tune generation params (optional)
        temperature = 0.2 if style == "technical" else 0.7  # more deterministic for technical, more open for casual

        # Return updates for the upcoming model call
        return {
            "messages": [{"role": "system", "content": system_prompt}],
            "model_kwargs": {"temperature": temperature},
        }


#### Create the agent

In [127]:

agent = create_agent(
    model,
    tools=tools,                       # e.g., [search, get_weather]
    middleware=[CustomMiddleware()],
    context_schema=Context,            # <-- use context, not state
    system_prompt="You are a helpful assistant. Be concise and accurate.",
)


#### Invoke the agent with user_preferences

In [128]:
query="Search for the explaination vector embeddings." 
query="Search for the story lines and theme in 三国演义 in Chinese" 
query="Search for the story lines and theme in Games of Throne and then introduce these in Chinese." 

In [129]:
# A user who prefers technical & detailed responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "technical",
            "verbosity": "detailed"
        }
    }
)
messages = result["messages"]

print("\n=============================== Assistant reply (technical + detailed) =================================")
print_message_pairs(messages,verbose=True)



=============================== Assistant reply (technical + detailed) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

**Storylines:**
1. **The Iron Throne:** The central storyline revolves around the struggle for power and control of the Iron Throne of the Seven Kingdoms. Various noble families, including the Starks, Lannisters, Baratheons, and Targaryens, vie for dominance.
2. **The Stark Family:** The Starks of Winterfell face numerous challenges, including betrayal, war, and the quest for justice. Key members like Ned Stark, Jon Snow, and Arya Stark play pivotal roles.
3. **Daenerys Targaryen's Quest:** Daenerys Targaryen's journey from exile to power, as she seeks to reclaim the throne and liberate the oppressed, is a major 

In [130]:

# A user who prefers casual & brief responses
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content":query
            }
        ],
    },
    context={
        "user_preferences": {
            "style": "casual",
            "verbosity": "brief"
        }
    }
)
messages = result["messages"]


print("\n=============================== Assistant reply (casual + brief) =================================")
print_message_pairs(messages,verbose=True)




=============================== Assistant reply (casual + brief) =================================
{
  "user_query": "Search for the story lines and theme in Games of Throne and then introduce these in Chinese.",
  "tool_name": "search",
  "tool_output": "Results for: Game of Thrones storylines and themes"
}

Aassistant reply:
**Game of Thrones Storylines and Themes:**

- **Storylines:**
  - **Power Struggles:** The series revolves around the battle for the Iron Throne among noble families.
  - **Family Dynamics:** Focuses on the relationships and conflicts within families like the Starks, Lannisters, and Targaryens.
  - **Mystical Elements:** Includes dragons, magic, and the threat of the White Walkers.
  - **Political Intrigue:** Features alliances, betrayals, and complex political maneuvers.

- **Themes:**
  - **Power and Ambition:** Explores the lengths people go to gain and maintain power.
  - **Loyalty and Betrayal:** Highlights the importance and consequences of loyalty and bet

#### Invoke the agent with user_preferences

#### Advanced concepts

##### ToolStrategy

ToolStrategy uses artificial tool calling to generate structured output. This works with any model that supports tool calling:

In [131]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    tools=[search],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


result["structured_response"]
# # ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

##### Memory

In [132]:
def demo_args(*args):
    print(args)  # args 是一个元组
    for i, value in enumerate(args, start=1):
        print(f"第{i}个参数: {value}")

import json 
demo_args(10, 20, 30)

(10, 20, 30)
第1个参数: 10
第2个参数: 20
第3个参数: 30


In [133]:
import json

def demo_kwargs(**kwargs):
    print(kwargs)  # kwargs 是一个字典
    for key1, value2 in kwargs.items():
        print(f"{key1} = {value2}")
   
    print(json.dumps(kwargs)) 
demo_kwargs(name="Alice", age=25, city="Shanghai")

{'name': 'Alice', 'age': 25, 'city': 'Shanghai'}
name = Alice
age = 25
city = Shanghai
{"name": "Alice", "age": 25, "city": "Shanghai"}


In [134]:
class MyClass:
    @staticmethod
    def static_method():
        print("This is a static method.")

    @classmethod
    def class_method(cls):
        print(f"This is a class method of {cls.__name__}.")

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value

# 使用
MyClass.static_method()
MyClass.class_method()

obj = MyClass()
obj.name = "Alice"
print(obj.name)

This is a static method.
This is a class method of MyClass.
Alice
